1. Из ноутбуков по практике "Рекуррентные и одномерные сверточные нейронные сети" выберите лучшую сеть, либо создайте свою.
2. Запустите раздел "Подготовка"
3. Подготовьте датасет с параметрами `VOCAB_SIZE=20'000`, `WIN_SIZE=1000`, `WIN_HOP=100`, как в ноутбуке занятия, и обучите выбранную сеть. Параметры обучения можно взять из практического занятия. Для  всех обучаемых сетей в данной работе они должны быть одни и теже.
4. Поменяйте размер словаря tokenaizera (`VOCAB_SIZE`) на `5000`, `10000`, `40000`.  Пересоздайте датасеты, при этом оставьте `WIN_SIZE=1000`, `WIN_HOP=100`.
Обучите выбранную нейронку на этих датасетах.  Сделайте выводы об  изменении  точности распознавания авторов текстов. Результаты сведите в таблицу
5. Поменяйте длину отрезка текста и шаг окна разбиения текста на векторы  (`WIN_SIZE`, `WIN_HOP`) используя значения (`500`,`50`) и (`2000`,`200`). Пересоздайте датасеты, при этом оставьте `VOCAB_SIZE=20000`. Обучите выбранную нейронку на этих датасетах. Сделайте выводы об  изменении точности распознавания авторов текстов.

Результаты всей работы сведите в таблицу.

## Подготовка

In [1]:
import os
import re
import copy
import numpy as np
import torch
import torch.nn as nn
import gdown
from tensorflow.keras.preprocessing.text import Tokenizer
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import torch.nn.functional as F

In [2]:
gdown.download(
    'https://storage.yandexcloud.net/aiueducation/Content/base/l7/writers.zip',
    None,
    quiet=True
)

!unzip -o writers.zip -d writers/

Archive:  writers.zip
  inflating: writers/(Клиффорд_Саймак) Обучающая_5 вместе.txt  
  inflating: writers/(Клиффорд_Саймак) Тестовая_2 вместе.txt  
  inflating: writers/(Макс Фрай) Обучающая_5 вместе.txt  
  inflating: writers/(Макс Фрай) Тестовая_2 вместе.txt  
  inflating: writers/(О. Генри) Обучающая_50 вместе.txt  
  inflating: writers/(О. Генри) Тестовая_20 вместе.txt  
  inflating: writers/(Рэй Брэдберри) Обучающая_22 вместе.txt  
  inflating: writers/(Рэй Брэдберри) Тестовая_8 вместе.txt  
  inflating: writers/(Стругацкие) Обучающая_5 вместе.txt  
  inflating: writers/(Стругацкие) Тестовая_2 вместе.txt  
  inflating: writers/(Булгаков) Обучающая_5 вместе.txt  
  inflating: writers/(Булгаков) Тестовая_2 вместе.txt  


In [3]:
# Папка с текстами авторов
FILE_DIR = "writers"

# Метки для определения обучающей и тестовой выборки
SIG_TRAIN = "обучающая"
SIG_TEST = "тестовая"

# Получаем список всех файлов и сортируем его
file_list = sorted(os.listdir(FILE_DIR))

CLASS_LIST = []

text_train = []

text_test = []

# Проходим по всем файлам
for file_name in file_list:

    # Извлекаем имя автора и тип выборки из названия файла
    m = re.match(r"\((.+)\) (\S+)_", file_name)

    if not m:
        continue

    # Имя автора
    class_name = m[1]

    # Тип выборки
    subset_name = m[2].lower()

    # Проверяем принадлежность к train/test
    is_train = SIG_TRAIN in subset_name
    is_test = SIG_TEST in subset_name

    # Если автор встречается впервые — добавляем его
    if class_name not in CLASS_LIST:

        CLASS_LIST.append(class_name)

        # Создаём пустые строки для текстов автора
        text_train.append("")
        text_test.append("")


    cls = CLASS_LIST.index(class_name)


    with open(
        os.path.join(FILE_DIR, file_name),
        "r",
        encoding="utf-8"
    ) as f:

        # Удаляем переносы строк
        text = f.read().replace("\n", " ")


    if is_train:
        text_train[cls] += " " + text

    elif is_test:
        text_test[cls] += " " + text

CLASS_COUNT = len(CLASS_LIST)

In [4]:
# Разбиваем  на окна фиксированной длины
def split_sequence(seq, win, hop):
    return [
        seq[i:i + win]
        for i in range(0, len(seq) - win + 1, hop)
        if len(seq[i:i + win]) == win
    ]
#строит словарь наиболее частых слов
def prepare_data(vocab_size, win_size, hop):

    tokenizer = Tokenizer(
        num_words=vocab_size,
        lower=True,
        oov_token="<UNK>"
    )

    tokenizer.fit_on_texts(text_train)
    # Преобразуем тексты в последовательности индексов
    train_seq = tokenizer.texts_to_sequences(text_train)
    test_seq = tokenizer.texts_to_sequences(text_test)

    x_train, y_train = [], []
    x_val, y_val = [], []
    # Разбиваем тексты на окна фиксированной длины
    for cls, seq in enumerate(train_seq):
        chunks = split_sequence(seq, win_size, hop)
        x_train.extend(chunks)
        # Каждому окну ставим метку автора
        y_train.extend([cls] * len(chunks))
    # Аналогично для тестовой выборки
    for cls, seq in enumerate(test_seq):
        chunks = split_sequence(seq, win_size, hop)
        x_val.extend(chunks)
        y_val.extend([cls] * len(chunks))

    return np.array(x_train), np.array(y_train), np.array(x_val), np.array(y_val)

In [8]:
class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, class_count, embed_dim=64, hidden_dim=128):
        super().__init__()

        # ереводит индексы слов в плотные векторы
        self.embedding = nn.Embedding(vocab_size, embed_dim)

        # зануляет целые каналы эмбеддингов
        self.spatial_dropout = nn.Dropout2d(0.3)

        # Двунаправленная GRU анализирует текст
        # одновременно слева направо и справа налево
        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        # Полносвязная часть классификатора
        self.fc = nn.Sequential(


            nn.Dropout(0.5),

            # Сжатие признаков
            nn.Linear(hidden_dim * 2, 64),


            nn.ReLU(),

            nn.Dropout(0.5),

            #
            nn.Linear(64, class_count)
        )

    def forward(self, x):

        x = self.embedding(x)

        x = x.permute(0, 2, 1).unsqueeze(-1)

        # Регуляризация эмбеддингов
        x = self.spatial_dropout(x)

        # Возвращаем исходный формат
        x = x.squeeze(-1).permute(0, 2, 1)

        # Выходы GRU для каждого шага последовательности
        out, _ = self.gru(x)

        # Max pooling выбирает наиболее значимые признаки
        x, _ = torch.max(out, dim=1)

        x = self.fc(x)

        return x

In [6]:
def train_model(model, train_loader, val_loader, verbose=False):

    criterion = nn.CrossEntropyLoss()
    # RMSprop хорошо работает с RNN/GRU сетями
    optimizer = torch.optim.RMSprop(
        model.parameters(),
        lr=1e-3,
        alpha=0.9
    )

    best_acc = 0
    best_state = None
    # Early stopping
    patience = 3
    patience_counter = 0

    for epoch in range(7):

        model.train()
        total_loss = 0

        for xb, yb in train_loader:

            xb, yb = xb.to(device), yb.to(device)

            optimizer.zero_grad()
            preds = model(xb)

            loss = criterion(preds, yb)
            loss.backward()

            optimizer.step()

            total_loss += loss.item()

        # Проверка на валидации
        model.eval()

        all_preds = []
        all_true = []

        with torch.no_grad():

            for xb, yb in val_loader:

                xb = xb.to(device)
                # Выбираем наиболее вероятный класс
                preds = model(xb)
                preds = torch.argmax(preds, dim=1)

                all_preds.extend(preds.cpu().numpy())
                all_true.extend(yb.numpy())

        acc = accuracy_score(all_true, all_preds)



        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    # Загружаем лучшие веса
    model.load_state_dict(best_state)

    return best_acc


In [13]:
def run_experiment(vocab_size, win_size, hop):

    # Подготавливаем датасеты
    # Тексты преобразуются в последовательности одинаковой длины
    x_train, y_train, x_val, y_val = prepare_data(
        vocab_size,
        win_size,
        hop
    )

    # Переводим данные в тензор
    x_train = torch.LongTensor(x_train)
    y_train = torch.LongTensor(y_train)

    x_val = torch.LongTensor(x_val)
    y_val = torch.LongTensor(y_val)

    # Фиксируем генератор случайных чисел
    # для воспроизводимости результатов
    g = torch.Generator()
    g.manual_seed(SEED)


    train_loader = DataLoader(

        # Объединяем признаки и метки
        TensorDataset(x_train, y_train),

        batch_size=32,

        shuffle=True,

        # Фиксированный генератор случайности
        generator=g
    )


    val_loader = DataLoader(

        TensorDataset(x_val, y_val),


        batch_size=32,

        # Для validation перемешивание не нужно
        shuffle=False
    )

    # Создание модели GRU
    model = GRUClassifier(


        vocab_size=vocab_size,


        class_count=CLASS_COUNT,


        embed_dim=64,

        # Размер скрытого состояния GRU
        hidden_dim=128

    ).to(device)

    # Обучение модели
    acc = train_model(
        model,
        train_loader,
        val_loader
    )


    return acc

In [14]:
experiments = [
    ("baseline", 20000, 1000, 100),
    ("vocab_5k", 5000, 1000, 100),
    ("vocab_10k", 10000, 1000, 100),
    ("vocab_40k", 40000, 1000, 100),
    ("win_small", 20000, 500, 50),
    ("win_large", 20000, 2000, 200),
]

results = []

for name, vocab, win, hop in experiments:

    acc = run_experiment(vocab, win, hop)

    results.append({
        "exp": name,
        "vocab": vocab,
        "win": win,
        "hop": hop,
        "val_accuracy": acc
    })

import pandas as pd
df = pd.DataFrame(results)
df

,exp,vocab,win,hop,val_accuracy
0,baseline,20000,1000,100,0.789036
1,vocab_5k,5000,1000,100,0.762916
2,vocab_10k,10000,1000,100,0.740385
3,vocab_40k,40000,1000,100,0.740528
4,win_small,20000,500,50,0.717124
5,win_large,20000,2000,200,0.707091


По результатам экспериментов можно сделать вывод, что для задачи распознавания авторов наилучшие результаты показала базовая конфигурация модели с параметрами VOCAB_SIZE = 20000, WIN_SIZE = 1000 и WIN_HOP = 100. Точность классификации в этом случае составила 0.789, что является лучшим результатом среди всех проведённых экспериментов.

Изменение размера словаря показало, что уменьшение количества слов ухудшает качество распознавания. При словаре размером 5000 точность снизилась до 0.763, а при 10000 — до 0.740. Это связано с тем, что для задачи распознавания авторов важны особенности словарного запаса и стиля текста, поэтому сокращение словаря приводит к потере значимой информации. Увеличение словаря до 40000 также не улучшило качество (0.741), так как большое количество редких слов добавляет шум и усложняет обучение модели.

Эксперименты с размером окна показали, что уменьшение окна до 500 символов или слов снижает точность до 0.717, поскольку модель получает меньше контекста для определения авторского стиля. Увеличение окна до 2000 также ухудшило результат (0.707), вероятно из-за избыточного контекста и появления большого количества менее значимых признаков.

Таким образом, для задачи распознавания авторов наиболее эффективной оказалась базовая конфигурация с размером словаря 20000 и размером окна 1000.